In [3]:
"Write program to implement KNN and K-means."

'Write program to implement KNN and K-means.'

In [2]:
import math
import random
from collections import Counter

# KNN classification and K-means clustering for the supplied dataset

hours_studied = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
attendance = [50, 55, 60, 65, 70, 75, 80, 85, 90, 95]
passed = [0, 0, 0, 0, 0, 1, 1, 1, 1, 1]

X = list(zip(hours_studied, attendance))
y = passed


def standardize(data):
    """Standardize each feature so features with larger scales don't dominate."""
    means = [sum(row[j] for row in data) / len(data) for j in range(len(data[0]))]
    stds = [
        (sum((row[j] - means[j]) ** 2 for row in data) / len(data)) ** 0.5 or 1
        for j in range(len(data[0]))
    ]
    scaled = [
        [(row[j] - means[j]) / stds[j] for j in range(len(means))]
        for row in data
    ]
    return scaled, means, stds


def scale_point(point, means, stds):
    return [(point[j] - means[j]) / stds[j] for j in range(len(point))]


def distance(a, b):
    return math.sqrt(sum((x - z) ** 2 for x, z in zip(a, b)))


# --- KNN: predict Pass (1) or Fail (0) ---
X_scaled, means, stds = standardize(X)


def knn_predict(query, k=3):
    query_scaled = scale_point(query, means, stds)
    neighbors = sorted(
        range(len(X_scaled)),
        key=lambda i: distance(query_scaled, X_scaled[i])
    )[:k]
    return Counter(y[i] for i in neighbors).most_common(1)[0][0]


query = [5.5, 72.5]
prediction = knn_predict(query, k=3)
print(f"KNN prediction for {query}: {'Pass' if prediction else 'Fail'}")


# --- K-means: cluster rows using Hours_Studied and Attendance only ---
def kmeans(data, k=2, max_iterations=100, seed=42):
    random.seed(seed)
    centers = [list(row) for row in random.sample(data, k)]

    for _ in range(max_iterations):
        clusters = [
            min(range(k), key=lambda c: distance(row, centers[c]))
            for row in data
        ]

        new_centers = []
        for cluster_id in range(k):
            members = [data[i] for i in range(len(data)) if clusters[i] == cluster_id]
            if not members:  # Keep the existing center if a cluster is empty.
                new_centers.append(centers[cluster_id])
            else:
                new_centers.append([
                    sum(row[j] for row in members) / len(members)
                    for j in range(len(data[0]))
                ])

        if new_centers == centers:
            break
        centers = new_centers

    return clusters, centers


cluster_ids, centers = kmeans(X_scaled, k=2)

print("\nK-means cluster assignments (cluster numbers are arbitrary):")
for row, cluster_id in zip(X, cluster_ids):
    print(f"Hours={row[0]}, Attendance={row[1]} -> Cluster {cluster_id}")

print("\nCluster centers (standardized features):")
for center in centers:
    print(center)

KNN prediction for [5.5, 72.5]: Fail

K-means cluster assignments (cluster numbers are arbitrary):
Hours=1, Attendance=50 -> Cluster 1
Hours=2, Attendance=55 -> Cluster 1
Hours=3, Attendance=60 -> Cluster 1
Hours=4, Attendance=65 -> Cluster 1
Hours=5, Attendance=70 -> Cluster 0
Hours=6, Attendance=75 -> Cluster 0
Hours=7, Attendance=80 -> Cluster 0
Hours=8, Attendance=85 -> Cluster 0
Hours=9, Attendance=90 -> Cluster 0
Hours=10, Attendance=95 -> Cluster 0

Cluster centers (standardized features):
[0.6963106238227913, 0.6963106238227913]
[-1.044465935734187, -1.044465935734187]
